# CH101 free AI 3D autobuild

This notebook creates non-production CH101 review candidates. It uses Stable Fast 3D as the default free Colab provider, with TripoSR as the free single-view fallback. Tripo API is optional and never required. It never enables Unity input or approves Gate B.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = 'CH101'
PROVIDER = os.environ.get('RE_CAMP_AI3D_PROVIDER', 'sf3d').lower()
MAX_ATTEMPTS = 3
CANDIDATE_COUNT = int(os.environ.get('RE_CAMP_AI3D_CANDIDATES', '4' if PROVIDER == 'tripo' else '1'))
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
# Set RE_CAMP_BLENDER_TOOLS_COMMIT for a reproducible pin; otherwise resolve the branch tip.
TOOLS_COMMIT = os.environ.get('RE_CAMP_BLENDER_TOOLS_COMMIT', '')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
CONTENT_ROOT = Path('/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidates' / PROVIDER
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
REVIEW_DIR = OUTPUT_ROOT / 'review'
assert PROVIDER in {'tripo', 'sf3d', 'triposr'}
print({'provider': PROVIDER, 'maxAttempts': MAX_ATTEMPTS, 'candidateCount': CANDIDATE_COUNT, 'output': str(OUTPUT_ROOT)})


In [ ]:
def run(command, **kwargs):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if shutil.which('blender') is None or shutil.which('xvfb-run') is None:
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'])
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
if TOOLS_COMMIT:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_COMMIT])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', TOOLS_COMMIT])
else:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
    TOOLS_COMMIT = subprocess.check_output(['git', '-C', TOOLS_DIR, 'rev-parse', 'HEAD'], text=True).strip()
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('blender:', shutil.which('blender'))
# Blender 3.0's bundled Python needs its own compatible NumPy wheel for GLB import.
BLENDER_PYTHON_SITE = CONTENT_ROOT / 'blender-python-site'
BLENDER_PYTHON_SITE.mkdir(parents=True, exist_ok=True)
if not (BLENDER_PYTHON_SITE / 'numpy').is_dir():
    run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--target', BLENDER_PYTHON_SITE,
        '--platform', 'manylinux_2_17_x86_64',
        '--python-version', '3.10',
        '--only-binary=:all:', '--no-deps', 'numpy==1.23.5',
    ])
BLENDER_ENVIRONMENT = os.environ.copy()
BLENDER_ENVIRONMENT['PYTHONPATH'] = str(BLENDER_PYTHON_SITE)


In [ ]:
prepare_script = TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py'
run([
    sys.executable, prepare_script,
    '--art-root', ART_DIR,
    '--output-dir', REFERENCE_DIR,
])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
reference_manifest = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
assert reference_manifest['artCommit'] == ART_COMMIT
assert reference_manifest['unityInputAllowed'] is False
print(json.dumps(reference_manifest, indent=2, ensure_ascii=False))


In [ ]:
CANDIDATE_MANIFESTS = []
attempt_summaries = []
provider_environment = os.environ.copy()
if PROVIDER == 'tripo':
    api_key = os.environ.get('TRIPO_API_KEY', '')
    try:
        from google.colab import userdata
        api_key = api_key or userdata.get('TRIPO_API_KEY')
    except Exception:
        pass
    command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'tripo_api.py',
        '--reference-manifest', REFERENCE_MANIFEST,
        '--output-dir', CANDIDATE_DIR,
        '--candidate-count', str(CANDIDATE_COUNT),
    ]
    if api_key:
        provider_environment['TRIPO_API_KEY'] = api_key
        command.append('--execute')
    else:
        print('TRIPO_API_KEY is absent: creating a zero-credit dry-run plan only.')
    run(command, env=provider_environment)
    candidate_manifest_path = CANDIDATE_DIR / 'candidate-manifest.json'
    if candidate_manifest_path.is_file():
        CANDIDATE_MANIFESTS.append(candidate_manifest_path)
else:
    contract = json.loads((TOOLS_DIR / 'contracts' / 'ch101_ai3d_free_pipeline_v001.json').read_text(encoding='utf-8'))
    for attempt in range(1, MAX_ATTEMPTS + 1):
        provider_attempts = [PROVIDER, 'instantmesh', 'triposr'] if PROVIDER == 'sf3d' else [PROVIDER]
        attempt_succeeded = False
        for attempt_provider in provider_attempts:
            provider_environment = os.environ.copy()
            if attempt_provider == 'sf3d':
                provider_key = 'stableFast3D'
            elif attempt_provider == 'instantmesh':
                provider_key = 'instantMesh'
            else:
                provider_key = 'tripoSR'
            provider_config = contract['providers'][provider_key]
            provider_repo = CONTENT_ROOT / f"provider-{attempt_provider}"
            attempt_output_dir = OUTPUT_ROOT / 'attempts' / f'{attempt:02d}' / 'candidates' / attempt_provider
            attempt_output_dir.mkdir(parents=True, exist_ok=True)
            if not (provider_repo / '.git').is_dir():
                run(['git', 'clone', provider_config['repository'], provider_repo])
            run(['git', '-C', provider_repo, 'fetch', 'origin', provider_config['commit']])
            run(['git', '-C', provider_repo, 'checkout', '--detach', provider_config['commit']])
            if attempt_provider == 'instantmesh':
                run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-lightning==2.1.2', 'gradio==3.41.2', 'huggingface-hub', 'einops', 'omegaconf', 'torchmetrics', 'webdataset', 'accelerate', 'tensorboard', 'PyMCubes', 'trimesh>=4.4.0', 'rembg', 'transformers==4.34.1', 'diffusers==0.20.2', 'bitsandbytes', 'imageio[ffmpeg]', 'xatlas', 'plyfile'])
                try:
                    import nvdiffrast  # noqa: F401
                except ImportError:
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/NVlabs/nvdiffrast/'])
            elif attempt_provider == 'sf3d':
                # Colab compatibility: replace the broken gpytoolbox source pin.
                sf3d_requirements = CONTENT_ROOT / 'sf3d-colab-requirements.txt'
                sf3d_lines = [line for line in (provider_repo / 'requirements.txt').read_text(encoding='utf-8').splitlines() if not line.startswith('gpytoolbox==')]
                sf3d_requirements.write_text('\n'.join(sf3d_lines) + '\n', encoding='utf-8')
                run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x==13.6.0'])
                run([sys.executable, '-m', 'pip', 'install', '-q', 'gpytoolbox==0.3.3'])
                run([sys.executable, '-m', 'pip', 'install', '-q', '-r', sf3d_requirements], cwd=provider_repo)
            else:
                run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf==2.3.0', 'einops==0.7.0', 'transformers==4.35.0', 'trimesh>=4.4.0', 'rembg', 'xatlas==0.0.9', 'moderngl==5.10.0'])
                run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', 'huggingface-hub==0.25.2'])
                import importlib.util
                if importlib.util.find_spec('onnxruntime') is None:
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'])
                try:
                    import torchmcubes  # noqa: F401
                except ImportError:
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/tatsy/torchmcubes.git'])
            try:
                from google.colab import userdata
                hf_token = userdata.get('HF_TOKEN')
                if hf_token:
                    provider_environment['HF_TOKEN'] = hf_token
            except Exception:
                pass
            try:
                provider_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_open_source_provider.py', '--provider', attempt_provider, '--provider-repo', provider_repo, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', attempt_output_dir, '--execute']
                reference_view = 'front'
                foreground_ratio = None
                if attempt_provider == 'triposr':
                    reference_views = provider_config.get('referenceViews', ['front'])
                    reference_view = reference_views[(attempt - 1) % len(reference_views)]
                    provider_command.extend(['--reference-view', reference_view])
                    foreground_ratios = provider_config.get('foregroundRatios', [0.85])
                    foreground_ratio = foreground_ratios[(attempt - 1) % len(foreground_ratios)]
                    provider_command.extend(['--foreground-ratio', str(foreground_ratio)])
                run(provider_command, env=provider_environment)
                manifest = attempt_output_dir / 'candidate-manifest.json'
                if not manifest.is_file():
                    raise RuntimeError(f'provider produced no candidate manifest: {manifest}')
                CANDIDATE_MANIFESTS.append(manifest)
                # Keep later attempts on the provider that actually succeeded.
                PROVIDER = attempt_provider
                attempt_summaries.append({'attempt': attempt, 'provider': attempt_provider, 'referenceView': reference_view, 'foregroundRatio': foreground_ratio, 'manifest': str(manifest), 'status': 'GENERATED'})
                attempt_succeeded = True
                break
            except (subprocess.CalledProcessError, RuntimeError) as error:
                print(f'Attempt {attempt} provider {attempt_provider} failed: {error}')
        if not attempt_succeeded:
            attempt_summaries.append({'attempt': attempt, 'status': 'FAILED'})
    if not CANDIDATE_MANIFESTS:
        raise RuntimeError('all AI candidate attempts failed')
print(json.dumps({'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS], 'attempts': attempt_summaries, 'unityInputAllowed': False}, indent=2, ensure_ascii=False))


In [ ]:
score_reports = []
refinement_reports = []
REFINEMENT_STATUS = 'REFINED_REVIEW_CANDIDATE'
face_driver_status = 'BLOCKED_NO_RELIABLE_FREE_FACE_LANDMARK_TRANSFER'
socket_review_status = 'AUTO_ESTIMATED_NOT_APPROVED'
candidate_manifests = [Path(path) for path in CANDIDATE_MANIFESTS if Path(path).is_file()]
if candidate_manifests:
    launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
    for manifest_path in candidate_manifests:
        candidate_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        assert candidate_manifest['unityInputAllowed'] is False
        attempt_label = manifest_path.parents[2].name if manifest_path.parent.name == 'triposr' or manifest_path.parent.name == 'sf3d' else 'attempt_00'
        attempt_number = int(attempt_label) if attempt_label.isdigit() else 0
        candidates = [entry for entry in candidate_manifest.get('candidates', []) if entry.get('status') == 'DOWNLOADED']
        for entry in candidates:
            source_candidate_id = entry['candidateId']
            candidate_id = f'{attempt_label}-{source_candidate_id}'
            candidate_output = EVALUATION_DIR / candidate_id
            candidate_output.mkdir(parents=True, exist_ok=True)
            refined_glb = candidate_output / f'{candidate_id}_refined.glb'
            refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
            refinement_report = candidate_output / 'refinement-report.json'
            try:
                run(launcher + [
                    'blender', '-b',
                    '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py',
                    '--',
                    '--candidate', entry['modelPath'],
                    '--output-glb', refined_glb,
                    '--output-blend', refined_blend,
                    '--report', refinement_report,
                    '--provider', candidate_manifest.get('provider', 'unknown'),
                    '--attempt', str(attempt_number),
                    '--parent-sha256', entry.get('sha256', ''),
                ], env=BLENDER_ENVIRONMENT)
                refinement_reports.append(refinement_report)
                evaluation_report = candidate_output / 'evaluation-report.json'
                normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
                run(launcher + [
                    'blender', '-b',
                    '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py',
                    '--',
                    '--candidate', refined_glb,
                    '--candidate-id', candidate_id,
                    '--output-dir', candidate_output,
                    '--report', evaluation_report,
                    '--normalized-blend', normalized_blend,
                ], env=BLENDER_ENVIRONMENT)
                score_report = candidate_output / 'candidate-score.json'
                run([
                    sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py',
                    '--reference-manifest', REFERENCE_MANIFEST,
                    '--evaluation-report', evaluation_report,
                    '--output', score_report,
                ])
                score_reports.append(score_report)
            except subprocess.CalledProcessError as error:
                print(f'Refinement/evaluation failed for {candidate_id}: {error}')
else:
    print('No downloaded candidates. Run the provider cell with a Colab GPU.')
print(json.dumps({'refinementReports': [str(path) for path in refinement_reports], 'scoreReports': [str(path) for path in score_reports], 'unityInputAllowed': False}, indent=2, ensure_ascii=False))


In [ ]:
RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
if score_reports:
    rank_command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py',
        '--output', RANKING_MANIFEST,
    ]
    for score_report in score_reports:
        rank_command.extend(['--score-report', score_report])
    run(rank_command)
    ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
    assert ranking['unityInputAllowed'] is False
    if ranking.get('selectedCandidate'):
        REVIEW_DIR.mkdir(parents=True, exist_ok=True)
        launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
        run(launcher + [
            'blender', '-b',
            '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ai3d_review_asset.py',
            '--',
            '--ranking-manifest', RANKING_MANIFEST,
            '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json',
            '--output-blend', REVIEW_DIR / 'CH101_AI_AutoReview_NOT_PRODUCTION_v001.blend',
            '--report', REVIEW_DIR / 'ai3d-review-report.json',
        ])
    else:
        print('All candidates are below threshold: regeneration is required.')
else:
    print('Ranking skipped because no candidate score reports exist.')


In [ ]:
archive_base = CONTENT_ROOT / f"re-camp-{CHARACTER_CODE}-ai3d-review-NOT-PRODUCTION"
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
print('Archive:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print('Browser download is unavailable; copy the archive before the session ends.')
